In [ ]:
!pip install -U google-genai sentence-transformers chromadb langchain-text-splitters pypdf

In [ ]:
import os
from google import genai
from google.colab import userdata

# Get API key from environment variable
api_key = os.environ.get("Gemini_API_Key")

# If not found, get it from Google Colab Secrets
if not api_key:
    try:
        api_key = userdata.get("Gemini_API_Key")
    except Exception:
        api_key = None

if not api_key:
    raise ValueError("Gemini_API_Key not found. Add it to Colab Secrets.")

# Create Gemini client
client = genai.Client(api_key=api_key)

print("Gemini client initialized successfully.")

Gemini client initialized successfully.


In [ ]:
from google.colab import files
from pypdf import PdfReader

print("Please upload one or more PDF files:")

uploaded = files.upload()

pdf_texts = []

for filename in uploaded.keys():

    if filename.lower().endswith(".pdf"):

        reader = PdfReader(filename)

        text = ""

        for page_num, page in enumerate(reader.pages):

            page_text = page.extract_text()

            if page_text:
                text += f"\n--- Page {page_num + 1} ---\n"
                text += page_text

        pdf_texts.append(text)

        print(f"Loaded `{filename}` ({len(reader.pages)} pages.)")


if not pdf_texts:
    raise ValueError("No PDF files found in upload.")

full_pdf_content = "\n\n".join(pdf_texts)

print("PDF text extraction complete.")

Please upload one or more PDF files:


Saving Windows Function ADBMS SQLPLUS(2).pdf to Windows Function ADBMS SQLPLUS(2).pdf
Loaded `Windows Function ADBMS SQLPLUS(2).pdf` (2 pages.)
PDF text extraction complete.


In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=100
)

chunks = text_splitter.split_text(full_pdf_content)

print(f"Extracted and split document into {len(chunks)} text chunks.")

Extracted and split document into 7 text chunks.


In [ ]:
print("Loading embedding model and building vector index...")

from sentence_transformers import SentenceTransformer
import chromadb

# Load embedding model
embedder = SentenceTransformer("all-MiniLM-L6-v2")

# Create ChromaDB client
chroma_client = chromadb.Client()

# Reset collection for clean execution
try:
    chroma_client.delete_collection(name="pdf_rag_collection")
except Exception:
    pass

# Create collection
collection = chroma_client.create_collection(
    name="pdf_rag_collection"
)

# Generate embeddings
chunk_embeddings = embedder.encode(chunks).tolist()

# Create unique IDs
chunks_ids = [
    f"doc_chunk_{i}"
    for i in range(len(chunks))
]

# Add documents and embeddings to ChromaDB
collection.add(
    documents=chunks,
    embeddings=chunk_embeddings,
    ids=chunks_ids
)

print("PDF Vector Indexing complete.")

Loading embedding model and building vector index...


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

PDF Vector Indexing complete.
